In [1]:
import os
import sys
from pathlib import Path

import lightning.pytorch as pl
import torch

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

sys.path.append("../src")

from hand_to_tex.datasets import HMELightningDataModule
from hand_to_tex.models import BaselineTransformer, HMELightningModule

pl.seed_everything(42, workers=True)

if torch.cuda.is_available():
    accelerator = "gpu"
    devices = 1
elif torch.backends.mps.is_available():
    accelerator = "mps"
    devices = 1
else:
    accelerator = "cpu"
    devices = 1

print(f"Accelerator: {accelerator}, devices: {devices}")

Seed set to 42


Accelerator: cpu, devices: 1


## 1. Vocabulary and DataLoaders


In [ ]:
data_root = "../data/simpler"
vocab_path = "../data/assets/simple.json"

dm = HMELightningDataModule(
    root=data_root,
    vocab_path=Path(vocab_path),
    processed=True,
    batch_size=4,
    num_workers=0,
    pin_memory=False,
)

## 2. Model Initialization

In [3]:
baseline_net = BaselineTransformer(
    vocab_size=len(dm.vocab),
    pad_idx=dm.vocab.PAD,
    d_model=128,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
)

model = HMELightningModule(model=baseline_net, vocab=dm.vocab, lr=3e-4)
print(
    f"Model Trainable Constraint count: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters"
)

Model Trainable Constraint count: 1,512,528 parameters


## 3. Lightning Trainer Execution

In [4]:
# num_epochs = 10

trainer = pl.Trainer(
    max_epochs=1,
    limit_train_batches=5,
    limit_val_batches=2,
    accelerator="auto",
    logger=False,
    enable_checkpointing=False,
)

trainer.fit(model=model, datamodule=dm)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


FileNotFoundError: [Errno 2] No such file or directory: '..data\\simpler\\train.pt'